In [7]:
import random
from qdrant_client import QdrantClient
import json
from google import genai
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=api_key)

random.seed(42)

client = QdrantClient("http://localhost:6333")
COLLECTION_NAME = "physics_rag_collection_lat_nuc"

class QAOutput(BaseModel):
    question: str=Field(..., description = "Question based on the abstract (It can include LaTeX equations)")
    question_type: str = Field(..., description="Question type: \"factual\", \"conceptual\", or \"summary\"")


def fetch_random_samples(limit=5):
    records, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        limit=10000,
        with_payload=True,
        with_vectors=False
    )

    # Abstract is long enough
    valid_records = [
        r for r in records
        if r.payload.get('abstract') and len(r.payload['abstract']) > 200
    ]

    selected = random.sample(valid_records, min(len(valid_records), limit))
    return selected


prompt_template = """
    You are a researcher in Nuclear Theory (nucl-th).
    Based ONLY on the provided research paper abstract, create a question.
    
    The question should be specific enough to be answered by the abstract.
    If the question involves math variables, use LaTeX notation.

    Paper Title: {title}
    Abstract: {abstract}
    """.strip()


def generate_question(abstract, title):
    prompt = prompt_template.format(abstract=abstract, title=title)

    try:
        response = gemini_client.models.generate_content(
            model="models/gemini-2.5-flash-lite",
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_schema": QAOutput,
            }
        )

        return json.loads(response.text)

    except Exception as e:
        print(f"Error generating a question: {e}")

In [8]:
import pandas as pd
from tqdm.auto import tqdm
import time

samples = fetch_random_samples(limit=1000)

dataset = []

for record in tqdm(samples):
    payload = record.payload
    abstract = payload['abstract']
    title = payload['title']
    preprint_date = payload['preprint_date']

    q = generate_question(abstract, title)

    if q:
        entry = {
            "paper_id": record.id,
            "title": title,
            "preprint_date": preprint_date,
            "question": q.get("question"),
            "question_type": q.get("question_type"),
        }
        dataset.append(entry)


    time.sleep(1)

df=pd.DataFrame(dataset)

  0%|          | 0/1000 [00:00<?, ?it/s]

Error generating a question: Expecting property name enclosed in double quotes: line 3 column 2030271 (char 2030401)
Error generating a question: Unterminated string starting at: line 2 column 15 (char 16)
Error generating a question: Expecting ',' delimiter: line 3 column 2030572 (char 2030635)


In [9]:
len(dataset)

997

In [10]:
df

,paper_id,title,preprint_date,question,question_type
0,1869694,Optimizing multilayer Bayesian neural networks...,2021-06-22,"What network structure, among single-layer, do...",factual
1,1807488,Baryons in the Gross-Neveu model in 1+1 dimens...,2020-07-16,What is the focus of the current work concerni...,summary
2,2616275,Three-pion effects in $K^0-\bar{K}^0$ mixing,2022-12-19,What is the primary challenge in calculating t...,conceptual
3,2158600,The Nuclear Physics of Neutron Stars,2022-09-29,What is the main input required to study the s...,factual
4,2104884,Non-equilibrium charmonium regeneration in str...,2022-06-30,"According to the study, what is the minimum va...",factual
...,...,...,...,...,...
992,2626397,Hybrid star phenomenology from the properties ...,2023-01-25,What is the specific mass of the black widow p...,factual
993,1853363,Exact solution of the Brueckner-Bethe-Goldston...,2021-07-26,What is the effect of using the angle-average ...,summary
994,2801953,Hybrid Isentropic Twin Stars,2024-06-24,How does the temperature behavior during the p...,factual
995,1862487,Fluctuations in the nuclear pasta phase,2021-05-07,What effect does the coexistence of different ...,conceptual


In [11]:
df.iloc[-1]

paper_id                                                   2811998
title            Kolmogorov-Arnold networks in nuclear binding ...
preprint_date                                           2024-07-30
question         What was the root mean square error achieved b...
question_type                                              factual
Name: 996, dtype: object

In [ ]:
df.to_csv("../data/ground_truth_dataset.csv", index=False)